In [ ]:
from tsai.all import *
import sklearn.metrics as skm
import sklearn.metrics as skm
from fastai.callback.tracker import SaveModelCallback
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
sns.set_theme(style="white")
sns.set_context("paper", font_scale=1.5)
plt.rcParams['font.family'] = 'Arial'
my_setup()


## 1. prepare data

In [ ]:
data_root = 'data'
label_col = 'class'

class_num = '4_class'
sp_mode = 's'

save_root = os.path.join('tsai_Models', f'{class_num}__{sp_mode}')
if not os.path.exists(save_root):
    os.makedirs(save_root)

train_val_data_path = os.path.join(f'data/{class_num}', f'train_val.csv')
test_data_path = os.path.join(f'data/{class_num}', f'test.csv')
train_val_data = pd.read_csv(train_val_data_path)
test_data = pd.read_csv(test_data_path)
train_val_data.drop(columns=[c for c in train_val_data.columns if c.startswith('p')], inplace=True)
test_data.drop(columns=[c for c in test_data.columns if c.startswith('p')], inplace=True)
print(train_val_data.shape, test_data.shape)


In [ ]:
random_seed = 42
train_data, val_data = train_test_split(train_val_data, test_size=5000, random_state=random_seed, stratify=train_val_data['class'])
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
print(train_data.shape, val_data.shape)


In [ ]:
X_train = train_data.drop(columns=['id', label_col]).values
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
# y_train = np.array([class2idx[c] for c in train_data[label_col].values])
y_train = train_data[label_col].values

X_val = val_data.drop(columns=['id', label_col]).values
X_val = X_val.reshape(X_val.shape[0], 1, X_val.shape[1])
# y_val = np.array([class2idx[c] for c in val_data[label_col].values])
y_val = val_data[label_col].values

X_test = test_data.drop(columns=['id', label_col]).values
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
# y_test = np.array([class2idx[c] for c in test_data[label_col].values])
y_test = test_data[label_col].values

X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

In [ ]:
X, y, splits = combine_split_data([X_train, X_val], [y_train, y_val])
splits

## 2. inspect data

In [ ]:
label_counts = collections.Counter(y_val)
print('Counts by label:', dict(label_counts))
print(f'Naive Accuracy: {100*max(label_counts.values())/sum(label_counts.values()):0.2f}%')

In [ ]:
bs = 512

tfms  = [None, [Categorize()]]
dsets = TSDatasets(X, y, tfms=tfms, splits=splits)
dls   = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=[bs, bs*2])
test_ds = dls.dataset.add_test(X_test, y_test)
test_dl = dls.new(test_ds)
dls.show_batch()

In [ ]:
dls.show_batch()

## 3. Archs comparison

In [ ]:
repeat_num = 3

archs = [
    # (FCN, {}), 
    (ResNet, {}), 
    # # (xresnet1d34, {}), 
    (ResCNN, {}), 
    (RNN, {'n_layers':1, 'bidirectional': False}),
    # (RNN, {'n_layers':2, 'bidirectional': False}),
    # (RNNPlus, {'n_layers':1, 'bidirectional': False}),
    # # (RNNPlus, {'n_layers':2, 'bidirectional': False}),
    # (GRU, {'n_layers':1, 'bidirectional': False}),
    # # (GRU, {'n_layers':2, 'bidirectional': False}),
    # (GRUPlus, {'n_layers':1, 'bidirectional': False}),
    # # (GRUPlus, {'n_layers':2, 'bidirectional': False}),
    (LSTM, {'n_layers':1, 'bidirectional': False}), 
    # (LSTM, {'n_layers':2, 'bidirectional': False}),
    # (LSTM, {'n_layers':3, 'bidirectional': False}),
    # (LSTMPlus, {'n_layers':1, 'bidirectional': False}),
    # (LSTMPlus, {'n_layers':2, 'bidirectional': False}),
    (LSTM, {'n_layers':1, 'bidirectional': True}), 
    # (LSTM, {'n_layers':2, 'bidirectional': True}), 
    # (LSTM, {'n_layers':3, 'bidirectional': True}),
    # (LSTMPlus, {'n_layers':1, 'bidirectional': True}),
    # (LSTMPlus, {'n_layers':2, 'bidirectional': True}),
    # (LSTM_FCN, {}), 
    # # (LSTM_FCN, {'shuffle': False}), 

    (InceptionTime, {}),
    # (InceptionTime, {

    # }),
    
    (XceptionTime, {}), 
    (XceptionTimePlus, {}),
    # (OmniScaleCNN, {}), 
    (mWDN, {}),
    # (mWDN, {
    #     'levels': 2,
    # }),
    # (mWDN, {
    #     'levels': 3,
    # }),
    # (mWDN, {
    #     'levels': 4,
    # }),
    (mWDNPlus, {}),
    # (mWDNPlus, {
    #     'levels': 2,
    # }),
    # (mWDNPlus, {
    #     'levels': 3,
    # }),  
    # (mWDNPlus, {
    #     'levels': 4,
    # }),
    # (mWDNPlus, {
    #     'levels': 4,
    #     'depth': 6,
    #     'ks': 40,
    #     'conv_dropout': 0.,
    #     'fc_dropout': 0.,
    #     'nf': 32,
    # }),
    # (mWDNPlus, {
    #     'levels': 3,
    #     'depth': 8,
    #     'ks': 40,
    #     'conv_dropout': 0.2,
    #     'fc_dropout': 0.0,
    #     'nf': 32,
    # }),
    (TST, {}),
    (TSTPlus, {}),

    # (TSPerceiver, {}),
    # (TSiTPlus, {}),
    # (PatchTST, {}),
    # (RNN_FCNPlus, {}),
    # (TransformerRNNPlus, {}),
    # (TransformerGRUPlus, {}),
    # (TransformerLSTMPlus, {}),

]

if os.path.exists(os.path.join(save_root, 'archs_results.csv')):
    results = pd.read_csv(os.path.join(save_root, 'archs_results.csv'))
else:
    results = pd.DataFrame(columns=[
        'arch',
        'repeat_id',
        'hyperparams', 
        'total_params', 
        'train_loss', 
        'valid_loss', 
        'val_accuracy', 
        'val_best_accuracy', 
        'test_accuracy', 
        'test_precision',
        'test_recall',
        'test_f1',
        'test_roc_auc', 
        'test_mcc',
        'time',
    ])
if os.path.exists(os.path.join(save_root, 'all_train_results.pkl')):
    with open(os.path.join(save_root, 'all_train_results.pkl'), 'rb') as f:
        all_train_results = pickle.load(f)
else:
    all_train_results = {}

if os.path.exists(os.path.join(save_root, 'all_test_results.json')):
    with open(os.path.join(save_root, 'all_test_results.json'), 'r') as f:
        all_test_results_json = json.load(f)
else:
    all_test_results_json = {}
if os.path.exists(os.path.join(save_root, 'all_test_results.pkl')):
    with open(os.path.join(save_root, 'all_test_results.pkl'), 'rb') as f:
        all_test_results = pickle.load(f)
else:
    all_test_results = {}

for i, (arch, k) in enumerate(archs):
    for repeat_id in range(repeat_num):
        model = create_model(arch, dls=dls, **k)
        name = f'{arch.__name__}__{repeat_id}'
        while name in all_test_results_json:
            name += '_'
        print(name)
        learn = Learner(dls, model, metrics=accuracy, model_dir=save_root, cbs=[ShowGraph(), SaveModelCallback(monitor='accuracy', fname=name)])
        start = time.time()

        lr_max = 0.001
        epoch = 100
        
        learn.fit_one_cycle(epoch, lr_max)
        elapsed = time.time() - start

        all_train_results[name] = learn.recorder.values

        vals = learn.recorder.values[-1]
        best_accuracy = max([item[2] for item in learn.recorder.values])
        
        model_best_epoch = learn.load(name)
        # eval on test set
        test_probas, test_targets, test_preds = learn.get_preds(dl=test_dl, with_decoded=True, save_preds=None, save_targs=None)
        test_accuracy = skm.accuracy_score(test_targets, test_preds)
        test_precision = skm.precision_score(test_targets, test_preds, average="macro")
        test_recall = skm.recall_score(test_targets, test_preds, average="macro")
        test_f1 = skm.f1_score(test_targets, test_preds, average="macro")
        test_roc_auc = skm.roc_auc_score(test_targets, test_probas, average="macro", multi_class="ovr")
        test_mcc = skm.matthews_corrcoef(test_targets, test_preds)
        test_confusion_matrix = skm.confusion_matrix(test_targets, test_preds)
        
        all_test_results_json[name] = {
            # 'test_probas': test_probas.tolist(),
            # 'test_targets': test_targets.tolist(),
            # 'test_preds': test_preds.tolist(),
            'hyper_params': k,
            'test_accuracy': test_accuracy,
            'test_precision': test_precision,
            'test_recall': test_recall,
            'test_f1': test_f1,
            'test_roc_auc': test_roc_auc,
            'test_mcc': test_mcc,
            # 'test_confusion_matrix': test_confusion_matrix.tolist(),
        }
        all_test_results[name] = {
            'hyper_params': k,
            'test_probas': test_probas,
            'test_targets': test_targets,
            'test_preds': test_preds,
            'test_accuracy': test_accuracy,
            'test_precision': test_precision,
            'test_recall': test_recall,
            'test_f1': test_f1,
            'test_roc_auc': test_roc_auc,
            'test_mcc': test_mcc,
            'test_confusion_matrix': test_confusion_matrix,
        }
        results.loc[len(results)] = [arch.__name__, repeat_id, k, count_parameters(model), vals[0], vals[1], vals[2], best_accuracy, test_accuracy, test_precision, test_recall, test_f1, test_roc_auc, test_mcc, int(elapsed)]
        results.sort_values(by='test_accuracy', ascending=False, kind='stable', ignore_index=True, inplace=True)
        clear_output()
        display(results)
        with open(os.path.join(save_root, 'all_train_results.pkl'), 'wb') as f:
            pickle.dump(all_train_results, f)
        with open(os.path.join(save_root, 'all_test_results.json'), 'w') as f:
            json.dump(all_test_results_json, f, indent=2)
        with open(os.path.join(save_root, 'all_test_results.pkl'), 'wb') as f:
            pickle.dump(all_test_results, f)
        results.to_csv(os.path.join(save_root, 'archs_results.csv'), index=False)

## 5. Figure S4: benchmark results

In [ ]:
data_root = 'data'
label_col = 'class'

class_num = '4_class'
sp_mode = 's'

save_root = os.path.join('tsai_Models', f'{class_num}__{sp_mode}')
results = pd.read_csv(os.path.join(save_root, 'archs_results.csv'))

results = pd.concat([results, pd.DataFrame({
    "arch": ["COSORK (AutoGluon)"],
    "test_accuracy": [0.6352],
    "test_mcc": [0.5162],
    "test_precision": [0.6274],
    "test_recall": [0.6352],
    "test_f1": [0.6269],
    "test_roc_auc": [0.8746],
})], ignore_index=True)

results = results.sort_values(by='test_accuracy', ascending=False)
results

In [ ]:
plt.figure(figsize=(4.5, 4), dpi=300)
sns.barplot(x='test_accuracy', y='arch', data=results, errorbar='ci', ci=95, palette='tab20')
plt.xlim(0.4, 0.65)
plt.ylabel('Model')
plt.xlabel('Test Accuracy with 95% CI')
plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold')
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1)
ax.tick_params(
    axis='both',
    which='major',
    direction='out',
    length=5,
    width=1,
    color='black',
    pad=5,
    bottom=True,  # Ensure ticks are drawn on the required edges
    left=True,    # Ensure ticks are drawn on the required edges
    top=False,    # Typically, ticks are not shown on the top and right
    right=False   # Color of the ticks (can be adjusted as needed)
)
ax.tick_params(
    axis='x', # Apply only to the X-axis
    which='minor',
    direction='out',
    length=2.5, # Minor ticks are usually shorter than major ticks
    width=0.7,  # Minor ticks are usually thinner than major ticks
    color='black', # Consistent or slightly lighter than major ticks
    bottom=True,
    top=False
)